In [2]:
import librosa 
import numpy as np 
from IPython.display import Audio

In [20]:
path = '/mindhive/mcdermott/www/mturk_stimuli/imgriff/single_word_rec/pilot'

In [55]:
!ls {path}/

PILOT_load_experiment_info.json  demo_words_16kHz  giga_200_words_16kHz
demo_words			 giga_200_words


In [54]:
## Convert all to 16kHz sampling rate

import os, sys 
from glob import glob

files = glob(path+'/*/*.wav')

for file in files:
    if 'demo_words' in file:
        new_path = file.replace('demo_words_original', 'demo_words')
    elif 'giga_200_words' in file:
        new_path = file.replace('giga_200_words_original', 'giga_200_words')
    cmd = f'ffmpeg -y -i {file} -ac 1 -ar 16000 {new_path}'
    try:
        os.system(cmd)
    except:
        print(f'Failed to run the cmd: {cmd}')
        


In [50]:
print(new_path)

/mindhive/mcdermott/www/mturk_stimuli/imgriff/single_word_rec/pilot/demo_words_16kHz/published.wav


In [67]:
demo_wavs = ['article.wav', 'university.wav', 'originally.wav']

demean = lambda x: x - x.mean()
rms_calc = lambda x: np.sqrt(np.mean(x**2))



In [78]:
# Get demo file 
demo_path = os.path.join(path, 'demo_words', demo_wavs[0])
wav,sr = librosa.load(demo_path, sr = None)



In [79]:
Audio(demo_path)

In [80]:
rms_demo = rms_calc(demean(wav))

In [81]:
rms_demo

0.0700001

In [103]:
# get trial file 

giga_words = glob(path+'/giga_200_words/*.wav')

g_wav, g_sr = librosa.load(giga_words[0], sr = None)


In [104]:
Audio(giga_words[0])

In [85]:
rms_giga = rms_calc(demean(g_wav))

In [86]:
rms_giga

0.0028661564

In [96]:
def convert_rms(x, new_rms):
    '''Convert x to new rms in dBFS'''
    rms_x = rms_calc(demean(x))
    curr_rms = 20*np.log10(rms_x)
    new_rms = 20*np.log10(new_rms)
    d = new_rms - curr_rms
    return x * 10**(d/20)

In [97]:
louder_eg = convert_rms(g_wav, rms_demo)

In [98]:
Audio(louder_eg, rate=g_sr)

In [99]:
rms_calc(demean(louder_eg))

0.070000075

In [101]:
import soundfile as sf

for wav_path in giga_words:
    wav, sr = librosa.load(wav_path, sr = None)
    new_rms_wav =  convert_rms(wav, rms_demo)
    sf.write(wav_path, new_rms_wav, sr, subtype='PCM_16')


In [102]:
wav

array([-3.0517578e-05, -3.0517578e-05, -3.0517578e-05, ...,
        0.0000000e+00,  0.0000000e+00,  0.0000000e+00], dtype=float32)